# Análisis de falsos negativos — OOF D-05 (protocolo C)

Caracterizamos los falsos negativos OOF del `random_forest` de D-05 (hojas `unhealthy` predichas `healthy`) según severidad, proxy de lesión, iluminación y cobertura de máscara Leaf COCO, sin reentrenar. Prefijo de resultados: `rocole_d07_`.


Librerías del cuaderno en una sola celda: tablas, arrays, figura, foto, COCO y conversión HSV.


In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pycocotools.coco import COCO
from skimage.color import rgb2hsv

Semilla fija del proyecto por si hace falta un orden auxiliar reproducible.


In [2]:
# Semilla fija del proyecto
seed = 42

seed

Rutas relativas a resultados D, OOF D-05, índice featured, HSV masked, JSON COCO Leaf y raíz del repo.


In [3]:
results_dir = Path("../../../experiments/results/rocole/D-cv-errores")
best_oof_path = results_dir / "rocole_d05_best_oof_predictions.csv"
index_path = Path("../../../data/featured/rocole/index.csv")
hsv_hist_path = Path("../../../data/featured/rocole/hsv_hist_features_masked.csv")
repo_root = Path("../../..").resolve()
coco_json_path = (
    repo_root / "data" / "raw" / "RoCoLe" / "Annotations" / "RoCoLE-coco.json"
)

print("repo_root:", repo_root)
print("results_dir existe:", results_dir.exists())
print("best_oof_path existe:", best_oof_path.exists())
print("index_path existe:", index_path.exists())
print("hsv_hist_path existe:", hsv_hist_path.exists())
print("coco_json_path existe:", coco_json_path.exists())
repo_root, results_dir, best_oof_path, index_path, hsv_hist_path, coco_json_path

repo_root: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe
results_dir existe: True
best_oof_path existe: True
index_path existe: True
hsv_hist_path existe: True
coco_json_path existe: True


Cargamos el OOF del ganador D-05 (`File`, `link_group`, `Binary.Label`, `Multiclass.Label`, `fold`, `y_true`, `y_pred`, `model_name`).


In [4]:
best_oof_predictions = pd.read_csv(best_oof_path)

print("filas OOF:", len(best_oof_predictions))
best_oof_predictions.head()

filas OOF: 1481


Confirmamos que el CSV trae solo `random_forest` y cuántas hojas OOF hay.


In [5]:
oof_model_name = best_oof_predictions["model_name"].unique()
n_oof_leaves = len(best_oof_predictions)

print("model_name en OOF:", oof_model_name)
print("solo random_forest:", set(oof_model_name) == {"random_forest"})
print("n_oof_leaves:", n_oof_leaves)
oof_model_name, n_oof_leaves

model_name en OOF: <StringArray>
['random_forest']
Length: 1, dtype: str
solo random_forest: True
n_oof_leaves: 1481


Nos quedamos con hojas enfermas reales del OOF (`y_true == unhealthy`).


In [6]:
unhealthy_oof_leaves = best_oof_predictions.loc[
    best_oof_predictions["y_true"] == "unhealthy"
].copy()

print("n_unhealthy:", len(unhealthy_oof_leaves))
unhealthy_oof_leaves.head()

n_unhealthy: 723


Cada enferma OOF es FN si `y_pred == healthy`; si no, TP.


In [7]:
unhealthy_oof_leaves["error_type"] = np.where(
    unhealthy_oof_leaves["y_pred"] == "healthy",
    "fn",
    "tp",
)

print(unhealthy_oof_leaves["error_type"].value_counts().to_dict())
unhealthy_oof_leaves.head()

{'tp': 574, 'fn': 149}


Apartamos la lista de falsos negativos hoja a hoja.


In [8]:
false_negative_leaves = unhealthy_oof_leaves.loc[
    unhealthy_oof_leaves["error_type"] == "fn"
].copy()

print("n_fn:", len(false_negative_leaves))
false_negative_leaves.head()

n_fn: 149


Conteo de FN y TP por `Multiclass.Label` (severidad/clase fina ya publicada).


In [9]:
fn_counts_by_multiclass = (
    unhealthy_oof_leaves.groupby("Multiclass.Label", as_index=False)
    .agg(
        n_leaves=("File", "count"),
        n_fn=("error_type", lambda s: (s == "fn").sum()),
        n_tp=("error_type", lambda s: (s == "tp").sum()),
    )
)

fn_counts_by_multiclass

Cargamos el índice featured para enlazar cada `File` con su `photo_path`.


In [10]:
rocole_index = pd.read_csv(index_path)

print("filas índice:", len(rocole_index))
print("columnas:", list(rocole_index.columns))
rocole_index.head()

filas índice: 1481
columnas: ['File', 'plant_key', 'link_group', 'Binary.Label', 'Multiclass.Label', 'leaf_face', 'split', 'photo_path']


Pegamos `photo_path` a las enfermas OOF por `File`.


In [11]:
unhealthy_oof_leaves = unhealthy_oof_leaves.merge(
    rocole_index[["File", "photo_path"]],
    on="File",
    how="left",
    validate="one_to_one",
)

print("sin photo_path:", unhealthy_oof_leaves["photo_path"].isna().sum())
unhealthy_oof_leaves.head()

sin photo_path: 0


Cargamos el JSON oficial Leaf de RoCoLe (mismo que B/C).


In [12]:
coco = COCO(str(coco_json_path))

print("imágenes en COCO:", len(coco.getImgIds()))
coco

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
imágenes en COCO: 1560


Función para máscara Leaf (OR de anotaciones; categoría Leaf), emparejando `File` por sufijo como en B-02. Sin fallback a foto completa.


In [13]:
def load_coco_leaf_mask(coco, file_name, height, width):
    """Máscara Leaf (OR) emparejando file_name del JSON por sufijo File."""
    # índice por sufijo File (URL COCO termina en -{File}); mismo criterio que endswith en B-02
    if not hasattr(load_coco_leaf_mask, "_by_file"):
        by_file = {}
        for img in coco.loadImgs(coco.getImgIds()):
            suffix = img["file_name"].rsplit("-", 1)[-1]
            by_file[suffix] = img
        load_coco_leaf_mask._by_file = by_file

    coco_image = load_coco_leaf_mask._by_file.get(file_name)
    if coco_image is None or not coco_image["file_name"].endswith(file_name):
        matched_images = [
            img
            for img in coco.loadImgs(coco.getImgIds())
            if img["file_name"].endswith(file_name)
        ]
        if len(matched_images) != 1:
            raise ValueError(
                f"esperaba 1 imagen COCO para {file_name}, hay {len(matched_images)}"
            )
        coco_image = matched_images[0]
    # categoría Leaf = 1; imgIds en lista (ids Labelbox son strings)
    ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
    if len(ann_ids) < 1:
        leaf_cat_ids = [
            cat["id"]
            for cat in coco.loadCats(coco.getCatIds())
            if cat.get("name", "").lower() == "leaf"
        ]
        ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=leaf_cat_ids)

    annotations = coco.loadAnns(ann_ids)
    if len(annotations) < 1:
        raise ValueError(f"sin anotaciones Leaf para {file_name}")

    leaf_mask = None
    for annotation in annotations:
        mask_i = coco.annToMask(annotation).astype(bool)
        leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

    if leaf_mask.shape != (height, width):
        raise ValueError(
            f"máscara {leaf_mask.shape} vs imagen {(height, width)} para {file_name}"
        )
    return leaf_mask


load_coco_leaf_mask

Banda verde fija alineada al bin `hsv_h_b2` del protocolo C (Hue skimage ∈ [0.25, 0.375)); el resto dentro de la máscara cuenta como no verde / proxy de mancha.


In [14]:
# Banda fija = bin hsv_h_b2 (8 bins H en [0, 1))
green_hue_low = 0.25
green_hue_high = 0.375

green_hue_low, green_hue_high

Con foto RGB y máscara Leaf calculamos tres proxies: fracción no verde, media de V y área relativa de máscara.


In [15]:
def compute_leaf_proxies(image, leaf_mask):
    """Proxies de lesión, iluminación y cobertura dentro de la máscara Leaf."""
    rgb = np.asarray(image, dtype=np.float64)
    if rgb.max() > 1.0:
        rgb = rgb / 255.0

    leaf_mask = np.asarray(leaf_mask, dtype=bool)
    if not np.any(leaf_mask):
        raise ValueError("máscara Leaf vacía")

    hsv = rgb2hsv(rgb)
    hue_values = hsv[:, :, 0][leaf_mask]
    value_values = hsv[:, :, 2][leaf_mask]

    green_pixels = (hue_values >= green_hue_low) & (hue_values < green_hue_high)
    non_green_fraction = float((~green_pixels).mean())
    mean_v = float(value_values.mean())
    mask_area_ratio = float(leaf_mask.mean())

    return {
        "non_green_fraction": non_green_fraction,
        "mean_v": mean_v,
        "mask_area_ratio": mask_area_ratio,
    }


compute_leaf_proxies

Recorremos solo enfermas OOF: leemos foto, máscara Leaf y proxies; si falla, NaN y mensaje en `proxy_error` (sin fallback a imagen completa).


In [16]:
np.random.seed(seed)
ordered_unhealthy = unhealthy_oof_leaves.sort_values("File").reset_index(drop=True)

proxy_rows = []
n_total = len(ordered_unhealthy)
for i_row, (_, row) in enumerate(ordered_unhealthy.iterrows(), start=1):
    proxy_row = {
        "File": row["File"],
        "non_green_fraction": np.nan,
        "mean_v": np.nan,
        "mask_area_ratio": np.nan,
        "proxy_error": None,
    }
    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        proxies = compute_leaf_proxies(sample_image, leaf_mask)
        proxy_row.update(proxies)
    except Exception as exc:
        proxy_row["proxy_error"] = str(exc)
    proxy_rows.append(proxy_row)
    if i_row % 100 == 0 or i_row == n_total:
        print(f"proxies {i_row}/{n_total}", flush=True)

unhealthy_leaf_proxies = pd.DataFrame(proxy_rows)

print("filas proxies:", len(unhealthy_leaf_proxies))
print("con proxy_error:", unhealthy_leaf_proxies["proxy_error"].notna().sum())
unhealthy_leaf_proxies.head()

proxies 100/723
proxies 200/723
proxies 300/723
proxies 400/723
proxies 500/723
proxies 600/723
proxies 700/723
proxies 723/723
filas proxies: 723
con proxy_error: 0


Pegamos los proxies calculados a las enfermas OOF.


In [17]:
unhealthy_leaves_with_proxies = unhealthy_oof_leaves.merge(
    unhealthy_leaf_proxies,
    on="File",
    how="left",
    validate="one_to_one",
)

print("filas:", len(unhealthy_leaves_with_proxies))
unhealthy_leaves_with_proxies.head()

filas: 723


Cargamos histogramas HSV masked solo para traer iluminación ya inventariada (`hsv_v_b0`…`hsv_v_b3`).


In [18]:
hsv_value_features = pd.read_csv(
    hsv_hist_path,
    usecols=["File", "hsv_v_b0", "hsv_v_b1", "hsv_v_b2", "hsv_v_b3"],
)

print("filas hsv_v:", len(hsv_value_features))
hsv_value_features.head()

filas hsv_v: 1481


Unimos `hsv_v_*` a la mesa de enfermas con proxies.


In [19]:
unhealthy_leaves_with_proxies = unhealthy_leaves_with_proxies.merge(
    hsv_value_features,
    on="File",
    how="left",
    validate="one_to_one",
)

print("filas:", len(unhealthy_leaves_with_proxies))
unhealthy_leaves_with_proxies[
    ["File", "error_type", "mean_v", "hsv_v_b0", "hsv_v_b1", "hsv_v_b2", "hsv_v_b3"]
].head()

filas: 723


Resumen numérico FN vs TP: media y mediana de `non_green_fraction` por `error_type` y por `Multiclass.Label`.


In [20]:
lesion_proxy_summary = (
    unhealthy_leaves_with_proxies.groupby(
        ["error_type", "Multiclass.Label"], as_index=False
    )
    .agg(
        n_leaves=("File", "count"),
        mean_non_green=("non_green_fraction", "mean"),
        median_non_green=("non_green_fraction", "median"),
    )
)

lesion_by_error = (
    unhealthy_leaves_with_proxies.groupby("error_type", as_index=False)
    .agg(
        n_leaves=("File", "count"),
        mean_non_green=("non_green_fraction", "mean"),
        median_non_green=("non_green_fraction", "median"),
    )
)
lesion_by_error["Multiclass.Label"] = "(all)"

lesion_proxy_summary = pd.concat(
    [lesion_by_error, lesion_proxy_summary],
    ignore_index=True,
)

lesion_proxy_summary

Figura de `non_green_fraction` (FN vs TP), con facetas por `error_type` y color por `Multiclass.Label`.


In [21]:
non_green_figure_path = results_dir / "rocole_d07_non_green_fraction_fn_vs_tp.png"

plot_leaves = unhealthy_leaves_with_proxies.dropna(subset=["non_green_fraction"]).copy()
multiclass_order = [
    "red_spider_mite",
    "rust_level_1",
    "rust_level_2",
    "rust_level_3",
    "rust_level_4",
]
facet_order = ["fn", "tp"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, error_type in zip(axes, facet_order):
    subset = plot_leaves.loc[plot_leaves["error_type"] == error_type]
    for label in multiclass_order:
        values = subset.loc[
            subset["Multiclass.Label"] == label, "non_green_fraction"
        ]
        if len(values) == 0:
            continue
        ax.hist(values, bins=15, alpha=0.45, label=label, range=(0, 1))
    ax.set_title(f"error_type = {error_type}")
    ax.set_xlabel("non_green_fraction")
    ax.legend(fontsize=7, title="Multiclass.Label")
axes[0].set_ylabel("Hojas")
fig.suptitle("Proxy de lesión (no verde) — FN vs TP")
fig.tight_layout()
fig.savefig(non_green_figure_path, dpi=150)
plt.close(fig)

print("figura:", non_green_figure_path.resolve())
non_green_figure_path

figura: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_non_green_fraction_fn_vs_tp.png


Resumen FN vs TP de iluminación: `mean_v` y, aparte, pesos en `hsv_v_b0`…`hsv_v_b3`.


In [22]:
illumination_proxy_summary = (
    unhealthy_leaves_with_proxies.groupby("error_type", as_index=False)
    .agg(
        n_leaves=("File", "count"),
        mean_mean_v=("mean_v", "mean"),
        median_mean_v=("mean_v", "median"),
        mean_hsv_v_b0=("hsv_v_b0", "mean"),
        mean_hsv_v_b1=("hsv_v_b1", "mean"),
        mean_hsv_v_b2=("hsv_v_b2", "mean"),
        mean_hsv_v_b3=("hsv_v_b3", "mean"),
    )
)

illumination_proxy_summary

Figura de iluminación (`mean_v` FN vs TP).


In [23]:
mean_v_figure_path = results_dir / "rocole_d07_mean_v_fn_vs_tp.png"

plot_leaves = unhealthy_leaves_with_proxies.dropna(subset=["mean_v"]).copy()

fig, ax = plt.subplots(figsize=(8, 4.5))
for error_type, group in plot_leaves.groupby("error_type"):
    ax.hist(
        group["mean_v"],
        bins=20,
        alpha=0.55,
        label=error_type,
        range=(0, 1),
    )
ax.set_xlabel("mean_v")
ax.set_ylabel("Hojas")
ax.set_title("Iluminación media (V) — FN vs TP")
ax.legend(title="error_type")
fig.tight_layout()
fig.savefig(mean_v_figure_path, dpi=150)
plt.close(fig)

print("figura:", mean_v_figure_path.resolve())
mean_v_figure_path

figura: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_mean_v_fn_vs_tp.png


Resumen FN vs TP de calidad/cobertura: `mask_area_ratio`.


In [24]:
mask_coverage_summary = (
    unhealthy_leaves_with_proxies.groupby("error_type", as_index=False)
    .agg(
        n_leaves=("File", "count"),
        mean_mask_area_ratio=("mask_area_ratio", "mean"),
        median_mask_area_ratio=("mask_area_ratio", "median"),
    )
)

mask_coverage_summary

Figura de `mask_area_ratio` (FN vs TP).


In [25]:
mask_figure_path = results_dir / "rocole_d07_mask_area_ratio_fn_vs_tp.png"

plot_leaves = unhealthy_leaves_with_proxies.dropna(subset=["mask_area_ratio"]).copy()

fig, ax = plt.subplots(figsize=(8, 4.5))
for error_type, group in plot_leaves.groupby("error_type"):
    ax.hist(
        group["mask_area_ratio"],
        bins=20,
        alpha=0.55,
        label=error_type,
        range=(0, 1),
    )
ax.set_xlabel("mask_area_ratio")
ax.set_ylabel("Hojas")
ax.set_title("Cobertura máscara Leaf — FN vs TP")
ax.legend(title="error_type")
fig.tight_layout()
fig.savefig(mask_figure_path, dpi=150)
plt.close(fig)

print("figura:", mask_figure_path.resolve())
mask_figure_path

figura: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_mask_area_ratio_fn_vs_tp.png


Guardamos la lista de FN con severidad y proxies.


In [26]:
false_negative_export = false_negative_leaves.merge(
    unhealthy_leaves_with_proxies[
        [
            "File",
            "non_green_fraction",
            "mean_v",
            "mask_area_ratio",
            "proxy_error",
            "hsv_v_b0",
            "hsv_v_b1",
            "hsv_v_b2",
            "hsv_v_b3",
            "photo_path",
        ]
    ],
    on="File",
    how="left",
    validate="one_to_one",
)

fn_csv_path = results_dir / "rocole_d07_false_negative_leaves.csv"
false_negative_export.to_csv(fn_csv_path, index=False)

print("n_fn guardados:", len(false_negative_export))
print("guardado:", fn_csv_path.resolve())
fn_csv_path

n_fn guardados: 149
guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_false_negative_leaves.csv


Guardamos la mesa completa de enfermas OOF (FN+TP) con proxies y los tres resúmenes (prefijo `rocole_d07_`; no tocamos A/B/C ni d05/d06).


In [27]:
unhealthy_proxies_path = results_dir / "rocole_d07_unhealthy_proxies.csv"
fn_by_multiclass_path = results_dir / "rocole_d07_fn_by_multiclass.csv"
proxy_summary_path = results_dir / "rocole_d07_proxy_summary_fn_vs_tp.csv"

unhealthy_leaves_with_proxies.to_csv(unhealthy_proxies_path, index=False)
fn_counts_by_multiclass.to_csv(fn_by_multiclass_path, index=False)

lesion_block = lesion_proxy_summary.copy()
lesion_block.insert(0, "proxy_family", "lesion_non_green")
illumination_block = illumination_proxy_summary.copy()
illumination_block.insert(0, "proxy_family", "illumination")
mask_block = mask_coverage_summary.copy()
mask_block.insert(0, "proxy_family", "mask_coverage")

proxy_summary_fn_vs_tp = pd.concat(
    [lesion_block, illumination_block, mask_block],
    ignore_index=True,
    sort=False,
)
proxy_summary_fn_vs_tp.to_csv(proxy_summary_path, index=False)

print("guardado:", unhealthy_proxies_path.resolve())
print("guardado:", fn_by_multiclass_path.resolve())
print("guardado:", proxy_summary_path.resolve())
unhealthy_proxies_path, fn_by_multiclass_path, proxy_summary_path

guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_unhealthy_proxies.csv
guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_fn_by_multiclass.csv
guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d07_proxy_summary_fn_vs_tp.csv


## Conclusiones

Sobre 723 enfermas OOF del `random_forest`, hay **149 FN** (y 574 TP). Dominan `rust_level_1` (109 FN / 333); el resto: 25 `rust_level_2`, 14 ácaro, 1 `rust_level_3`, 0 en `rust_level_4`.

El proxy de lesión separa: `non_green_fraction` media FN **0.42** vs TP **0.57** (medianas 0.30 vs 0.55). Iluminación (`mean_v` ≈ 0.63 vs 0.61) y cobertura Leaf (`mask_area_ratio` ≈ 0.27 en ambos) casi no distinguen FN de TP. Sin errores de máscara/proxy. No reentrenamos aquí.
